In [0]:
import sys
from pathlib import Path

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = Path("/Workspace") / Path(notebook_path).parent
except Exception:
    repo_root = Path.cwd()

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from config import (
    GOLD_MONTHLY_CONTENT_TABLE,
    SILVER_OTT_CONTENTS_TABLE,
    SILVER_OTT_LOGS_TABLE,
    SILVER_OTT_USERS_TABLE,
    genre_top3_table_name,
)

# ── 1. 원본 읽기 ──────────────────────────────────────────

df_logs = spark.table(SILVER_OTT_LOGS_TABLE)
df_users = spark.table(SILVER_OTT_USERS_TABLE)
df_contents = spark.table(SILVER_OTT_CONTENTS_TABLE)


# display(df_logs.limit(2))
# display(df_users.limit(2))
# display(df_contents.limit(2))

In [0]:
from pyspark.sql.functions import sum, col, rank, current_date, concat, lit, floor, date_format
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ── 2. 국가별 월별 컨텐츠 시청시간 집계 ──────────────────────────────────────────
df_monthly = (
    df_logs
    .join(df_users, on="user_id", how="inner")
    .join(df_contents.select("content_id", "genre"), on="content_id", how="inner")
    
    .withColumn("month", date_format(col("view_timestamp"), "yyyy-MM"))
    .groupBy("country", "month", "content_id", "genre")
    .agg(
        sum("watched_seconds").alias("total_watched_sec"),
        sum(col("watched_seconds") / 3600).cast("int").alias("total_watched_hours"),
    )
)


# ── 3. 적재 ──────────────────────────────────────────
if not spark.catalog.tableExists(GOLD_MONTHLY_CONTENT_TABLE):
    df_monthly.write.saveAsTable(GOLD_MONTHLY_CONTENT_TABLE)
else:
    target_df = DeltaTable.forName(spark, GOLD_MONTHLY_CONTENT_TABLE)
    
    (
        target_df.alias("target")
        .merge(
            df_monthly.alias("source"),
            """target.content_id = source.content_id
            AND target.month = source.month
            AND target.genre = source.genre
            AND target.country = source.country 
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()

    )
    
    

In [0]:
# ── 4. 이번달  ──────────────────────────────────────────

month = dbutils.widgets.get('month')
month_safe = month.replace("-", "_")
country = dbutils.widgets.get('country')

df_monthly.createOrReplaceTempView("df_monthly")
kr_genre_top3 = (
    spark.sql(
        f"""
            SELECT country, genre, total_watched_hours
            FROM (
                SELECT country, genre, SUM(total_watched_hours) AS total_watched_hours,
                    ROW_NUMBER() OVER (PARTITION BY country ORDER BY SUM(total_watched_sec) DESC) AS rnk
                FROM df_monthly
                WHERE month = '{month}'
                GROUP BY country, genre
            ) ranked
            WHERE rnk <= 3
            AND country = '{country}'
            ORDER BY country, rnk
        """
    )
)

(
    kr_genre_top3.write
    .mode("overwrite")
    .saveAsTable(genre_top3_table_name(country, month_safe))
)